# Módulo 21 · Práctica: aprendizaje por refuerzo y preferencias

Vas a recorrer, en miniatura, el camino que va de REINFORCE a DPO: el mismo que se usa para alinear un LLM con lo que la gente prefiere. Primero un bandido en NumPy, después un modelo de recompensa de Bradley-Terry y, por último, DPO con una política diminuta en PyTorch (CPU).

| Parte | Qué haces |
| --- | --- |
| 0 | Preparación |
| 1 | Bandido de 4 brazos con REINFORCE (NumPy) |
| 2 | Línea base y varianza del estimador |
| 3 | Modelo de recompensa de Bradley-Terry (NumPy) |
| 4 | Pares de preferencias sintéticos y una política diminuta (PyTorch) |
| 5 | DPO: entrenar y comprobar que suben las respuestas preferidas |
| 6 | Retos |

Las celdas `# TU CÓDIGO` son ejercicios; ejecuta la `# Comprobación` que las sigue. Tiempo de ejecución total: menos de un minuto en CPU.

## Parte 0 · Preparación

In [ ]:
import time, itertools
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)   # estabilidad numérica (módulo 11)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def sigmoide(x):
    return 1 / (1 + np.exp(-x))

T0 = time.time()
print("NumPy", np.__version__, "· PyTorch", torch.__version__)

## Parte 1 · Bandido de 4 brazos con REINFORCE

Un bandido es un MDP de un solo paso: eliges una acción a, recibes una recompensa r y se acaba el episodio. Para un LLM sería elegir una respuesta completa y recibir una nota.

La política es una softmax sobre logits θ ∈ ℝ⁴: $\pi_\theta(a) = \mathrm{softmax}(\theta)_a$. El truco log-derivada da

$$\nabla_\theta J = \mathbb{E}_{a\sim\pi_\theta}\big[\,r(a)\,\nabla_\theta \log \pi_\theta(a)\,\big], \qquad \nabla_\theta \log \pi_\theta(a) = e_a - \pi_\theta.$$

In [ ]:
MEDIAS = np.array([0.2, 0.5, 1.0, 0.3])   # recompensa media de cada brazo (el 2 es el mejor)
K = len(MEDIAS)

def tirar(a, rng):
    """Recompensa ruidosa del brazo a."""
    return MEDIAS[a] + rng.normal(0, 1.0)

**Ejercicio 1.** Implementa `grad_log_pi(theta, a)`, que devuelve el vector $e_a - \mathrm{softmax}(\theta)$.

In [ ]:
def grad_log_pi(theta, a):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
th = np.array([0.0, 0.0, 0.0])
assert grad_log_pi(th, 1) is not None, "Devuelve un array"
assert np.allclose(grad_log_pi(th, 1), [-1/3, 2/3, -1/3]), "Con θ = 0 y a = 1 debe salir (−1/3, 2/3, −1/3)"
# comparación con diferencias finitas de log π(a)
th = np.array([0.3, -1.0, 2.0, 0.5]); h = 1e-6
num = np.array([(np.log(softmax(th + h*np.eye(4)[k])[2]) - np.log(softmax(th - h*np.eye(4)[k])[2])) / (2*h) for k in range(4)])
assert np.allclose(grad_log_pi(th, 2), num, atol=1e-6), "No coincide con el gradiente numérico"
print("Correcto")

**Ejercicio 2.** Completa el bucle de REINFORCE. En cada paso: muestrea a ~ π, obtén r, calcula la ventaja (r − b si hay línea base, r si no) y sube por el gradiente: θ ← θ + lr · ventaja · ∇log π(a). La línea base b es una media móvil de las recompensas.

In [ ]:
def reinforce(n_pasos=2000, lr=0.1, linea_base=True, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros(K); b = 0.0; historia = []
    for t in range(n_pasos):
        p = softmax(theta)
        a = rng.choice(K, p=p)
        r = tirar(a, rng)
        # TU CÓDIGO: ventaja y actualización de theta
        if linea_base:
            b += 0.05 * (r - b)
        historia.append(p @ MEDIAS)     # recompensa esperada de la política actual
    return theta, np.array(historia)

In [ ]:
# Comprobación
finales = [softmax(reinforce(seed=s)[0])[2] for s in range(5)]
print("π(mejor brazo) tras 2000 pasos:", np.round(finales, 3))
assert min(finales) > 0.9, "La política debería concentrarse en el brazo 2 (π > 0,9)"
print("Correcto")

In [ ]:
for lb, nombre in [(False, "sin línea base"), (True, "con línea base")]:
    _, h = reinforce(linea_base=lb, seed=3)
    plt.plot(h, label=nombre)
plt.axhline(MEDIAS.max(), ls="--", c="gray"); plt.xlabel("paso"); plt.ylabel("E[r] de la política")
plt.legend(); plt.show()

## Parte 2 · Línea base y varianza

Restar una línea base b no cambia la esperanza del estimador, porque $\mathbb{E}[\nabla\log\pi]=0$, pero puede reducir mucho su varianza. Lo comprobamos en θ = 0 cuando todas las recompensas se desplazan +5 (algo habitual: un modelo de recompensa no tiene cero natural).

**Ejercicio 3.** Completa `muestras_gradiente`: devuelve una matriz (n, K) con una estimación $(r-b)\,\nabla\log\pi(a)$ por fila.

In [ ]:
def muestras_gradiente(b, n=20000, desplazamiento=5.0, seed=1):
    rng = np.random.default_rng(seed)
    theta = np.zeros(K); p = softmax(theta)
    G = np.zeros((n, K))
    for i in range(n):
        a = rng.choice(K, p=p)
        r = tirar(a, rng) + desplazamiento
        # TU CÓDIGO: G[i] = ...
    return G

In [ ]:
# Comprobación
exacto = 0.25 * (MEDIAS - MEDIAS.mean())        # ∇J en θ = 0: π_k (r̄_k − J)
G0, Gb = muestras_gradiente(0.0), muestras_gradiente(5.0 + MEDIAS.mean())
print("exacto      ", np.round(exacto, 3))
print("media b = 0 ", np.round(G0.mean(0), 3), "· varianza total", round(G0.var(0).sum(), 2))
print("media b = J ", np.round(Gb.mean(0), 3), "· varianza total", round(Gb.var(0).sum(), 2))
assert np.allclose(Gb.mean(0), exacto, atol=0.03), "Con línea base la media debe acercarse al gradiente exacto"
assert G0.var(0).sum() > 10 * Gb.var(0).sum(), "La línea base debería reducir la varianza más de 10 veces"
print("Correcto")

## Parte 3 · Modelo de recompensa de Bradley-Terry

Cada respuesta se describe con 5 rasgos (piensa en «es correcta», «es educada», «es larga»…). Unos anotadores simulados prefieren A sobre B con probabilidad $\sigma(r_A - r_B)$, donde $r = w^\top x$ con un $w$ verdadero oculto. Aprende $w$ minimizando

$$\mathcal{L}(w) = -\frac{1}{n}\sum_i \log\sigma\big(w^\top(x^{(i)}_w - x^{(i)}_l)\big).$$

In [ ]:
rng = np.random.default_rng(0)
d = 5
W_VERDADERO = np.array([2.0, -1.0, 0.5, 0.0, 1.5])
n = 600
XA, XB = rng.normal(size=(n, d)), rng.normal(size=(n, d))
prefiere_A = rng.random(n) < sigmoide((XA - XB) @ W_VERDADERO)
Xw = np.where(prefiere_A[:, None], XA, XB)     # respuesta ganadora
Xl = np.where(prefiere_A[:, None], XB, XA)     # respuesta perdedora
tr, te = slice(0, 500), slice(500, 600)
print(Xw.shape, "· fracción de veces que ganó A:", prefiere_A.mean())

**Ejercicio 4.** Implementa `bt_perdida_grad(w, Xw, Xl)`: devuelve la pérdida media y su gradiente. Pista: con m = w·(x_w − x_l), la derivada de −log σ(m) respecto a m es −(1 − σ(m)). Para la pérdida usa `np.logaddexp(0, -m)`, que es −log σ(m) estable.

In [ ]:
def bt_perdida_grad(w, Xw, Xl):
    # TU CÓDIGO
    perdida, grad = None, None
    return perdida, grad

In [ ]:
# Comprobación
L0, g0 = bt_perdida_grad(np.zeros(d), Xw[tr], Xl[tr])
assert abs(L0 - np.log(2)) < 1e-9, "Con w = 0 la pérdida debe ser ln 2 = 0,693"
w_t = np.array([0.1, 0.2, -0.3, 0.4, 0.0]); h = 1e-6
num = np.array([(bt_perdida_grad(w_t + h*np.eye(d)[k], Xw, Xl)[0] - bt_perdida_grad(w_t - h*np.eye(d)[k], Xw, Xl)[0]) / (2*h) for k in range(d)])
assert np.allclose(bt_perdida_grad(w_t, Xw, Xl)[1], num, atol=1e-6), "Gradiente incorrecto"
w = np.zeros(d)
for it in range(500):
    L, g = bt_perdida_grad(w, Xw[tr], Xl[tr])
    w -= 0.5 * g
acc = np.mean((Xw[te] - Xl[te]) @ w > 0)
cos = w @ W_VERDADERO / np.linalg.norm(w) / np.linalg.norm(W_VERDADERO)
print("w aprendido:", np.round(w, 2), "· pérdida", round(L, 3), "· acierto en test", acc, "· coseno con w verdadero", round(cos, 4))
assert acc > 0.75 and cos > 0.99, "El modelo de recompensa debería recuperar la dirección de w"
print("Correcto")

Fíjate: el acierto en test no llega al 100 % ni con el $w$ verdadero, porque las preferencias humanas (aquí simuladas) son ruidosas. Y $w$ solo está determinado salvo una constante aditiva en r: Bradley-Terry ve diferencias.

## Parte 4 · Pares de preferencias y una política diminuta

Mundo de juguete: hay 3 *prompts* y la respuesta son 2 tokens de un vocabulario de 5 palabras. La política es una tabla de logits indexada por (prompt, token anterior): un «LLM» con contexto de un token. La recompensa oculta premia la cortesía y castiga el insulto.

In [ ]:
VOCAB = ["gracias", "por favor", "vale", "no", "idiota"]
V, L, P = len(VOCAB), 2, 3        # vocabulario, longitud de respuesta, número de prompts
BOS = V                           # índice extra para «inicio de respuesta»

def recompensa(y):
    y = list(y)
    return y.count(0) + y.count(1) - 2 * y.count(4)

class Politica(nn.Module):
    def __init__(self):
        super().__init__()
        self.tabla = nn.Embedding(P * (V + 1), V)       # logits por (prompt, token previo)
        nn.init.normal_(self.tabla.weight, std=0.5)
    def logp(self, x, y):
        """log π(y | x) sumando los log-probs de cada token. x: (B,), y: (B, L)."""
        prev = torch.cat([torch.full_like(y[:, :1], BOS), y[:, :-1]], dim=1)
        logits = self.tabla(x[:, None] * (V + 1) + prev)
        return torch.log_softmax(logits, -1).gather(-1, y[..., None]).squeeze(-1).sum(-1)
    @torch.no_grad()
    def muestrear(self, x):
        prev = torch.full_like(x, BOS); toks = []
        for t in range(L):
            tok = torch.multinomial(F.softmax(self.tabla(x * (V + 1) + prev), -1), 1).squeeze(1)
            toks.append(tok); prev = tok
        return torch.stack(toks, 1)

torch.manual_seed(0)
ref = Politica().requires_grad_(False)          # modelo de referencia (el «SFT»), congelado
print(ref.muestrear(torch.tensor([0, 1, 2])).tolist())

In [ ]:
# Pares (y1, y2) muestreados de la referencia; un «anotador» elige con Bradley-Terry sobre la recompensa oculta
N = 2000
x = torch.randint(0, P, (N,))
y1, y2 = ref.muestrear(x), ref.muestrear(x)
r1 = torch.tensor([recompensa(a) for a in y1.tolist()], dtype=torch.float)
r2 = torch.tensor([recompensa(a) for a in y2.tolist()], dtype=torch.float)
gen = torch.Generator().manual_seed(1)
gana1 = torch.rand(N, generator=gen) < torch.sigmoid(r1 - r2)
yw = torch.where(gana1[:, None], y1, y2)
yl = torch.where(gana1[:, None], y2, y1)
xtr, ywtr, yltr = x[:1600], yw[:1600], yl[:1600]
xte, ywte, ylte = x[1600:], yw[1600:], yl[1600:]
for i in range(3):
    print(f"prompt {x[i].item()}:  preferida {[VOCAB[t] for t in yw[i]]}  ·  rechazada {[VOCAB[t] for t in yl[i]]}")

## Parte 5 · DPO

La pérdida de DPO (Rafailov et al., 2023) es una Bradley-Terry cuya «recompensa» es la recompensa implícita $\hat r(x,y)=\beta\log\frac{\pi_\theta(y|x)}{\pi_{\mathrm{ref}}(y|x)}$:

$$\mathcal{L}_{\mathrm{DPO}} = -\mathbb{E}\Big[\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}\Big)\Big].$$

**Ejercicio 5.** Implementa `perdida_dpo`. Devuelve la pérdida media y la fracción de pares con margen positivo. Usa `F.logsigmoid` (estable).

In [ ]:
def perdida_dpo(pol, ref, x, yw, yl, beta):
    # TU CÓDIGO
    perdida, acierto = None, None
    return perdida, acierto

In [ ]:
# Comprobación (antes de entrenar, π = π_ref)
pol = Politica(); pol.load_state_dict(ref.state_dict())
l0, a0 = perdida_dpo(pol, ref, xte, ywte, ylte, beta=0.5)
assert l0 is not None and abs(l0.item() - np.log(2)) < 1e-5, "Con π = π_ref todos los márgenes son 0 y la pérdida es ln 2"
assert l0.requires_grad, "La pérdida debe depender de los parámetros de pol"
print("Correcto: pérdida inicial =", round(l0.item(), 4))

In [ ]:
BETA = 0.5
opt = torch.optim.Adam(pol.parameters(), lr=0.05)
curva = []
for paso in range(150):
    idx = torch.randint(0, 1600, (256,))
    loss, _ = perdida_dpo(pol, ref, xtr[idx], ywtr[idx], yltr[idx], BETA)
    opt.zero_grad(); loss.backward(); opt.step()
    curva.append(loss.item())
plt.plot(curva); plt.xlabel("paso"); plt.ylabel("pérdida DPO"); plt.show()
with torch.no_grad():
    l1, a1 = perdida_dpo(pol, ref, xte, ywte, ylte, BETA)
print("test: pérdida", round(l1.item(), 3), "· acierto de preferencia", round(a1.item(), 3))

**Ejercicio 6.** La respuesta tiene solo 5² = 25 posibilidades, así que podemos calcular todo **exactamente**. Para cada prompt, calcula la recompensa esperada $\mathbb{E}_{y\sim\pi}[r(y)]$ bajo `ref` y bajo `pol`, y la de la política óptima teórica $\pi^*(y|x)\propto\pi_{\mathrm{ref}}(y|x)\,e^{r(y)/\beta}$. Guarda tres listas de longitud P: `Er_ref`, `Er_pol`, `Er_opt`.

In [ ]:
TODAS = torch.tensor(list(itertools.product(range(V), repeat=L)))          # (25, 2)
R_TODAS = torch.tensor([recompensa(a) for a in TODAS.tolist()], dtype=torch.float)

In [ ]:
Er_ref, Er_pol, Er_opt = [], [], []
with torch.no_grad():
    for p_ in range(P):
        X = torch.full((len(TODAS),), p_)
        # TU CÓDIGO
        pass

In [ ]:
# Comprobación
print("E[r]  ref:", np.round(Er_ref, 2), "· DPO:", np.round(Er_pol, 2), "· π* teórica:", np.round(Er_opt, 2))
assert len(Er_pol) == P, "Necesitas un valor por prompt"
assert all(b > a + 1.0 for a, b in zip(Er_ref, Er_pol)), "DPO debería subir E[r] más de 1 punto en cada prompt"
assert all(b < c + 0.3 for b, c in zip(Er_pol, Er_opt)), "DPO no debería superar claramente a π* (es el óptimo del objetivo con KL)"
with torch.no_grad():
    sube_w = (pol.logp(xte, ywte).exp().mean() / ref.logp(xte, ywte).exp().mean()).item()
    sube_l = (pol.logp(xte, ylte).exp().mean() / ref.logp(xte, ylte).exp().mean()).item()
print(f"P media de las preferidas ×{sube_w:.2f} · de las rechazadas ×{sube_l:.2f}")
assert sube_w > 1.2 and sube_l < 0.8, "Deben subir las preferidas y bajar las rechazadas"
print("Correcto")

In [ ]:
with torch.no_grad():
    X = torch.zeros(len(TODAS), dtype=torch.long)
    pr, pp = ref.logp(X, TODAS).exp(), pol.logp(X, TODAS).exp()
orden = torch.argsort(pp, descending=True)[:6]
for i in orden.tolist():
    print(f"{' '.join(VOCAB[t] for t in TODAS[i]):22s} r = {R_TODAS[i]:+.0f}   π_ref = {pr[i]:.3f}   π_DPO = {pp[i]:.3f}")
print(f"Tiempo total: {time.time() - T0:.1f} s")

## Parte 6 · Retos (sin comprobación automática)

1. **Efecto de β.** Repite la Parte 5 con β = 0,1 y β = 2. ¿Cómo cambian E[r] y la KL $\mathbb{E}_{\pi}[\log \pi/\pi_{\mathrm{ref}}]$? Relaciónalo con la sección 21.3.
2. **Reward hacking.** Cambia la recompensa por un *proxy* que solo cuente «gracias». Entrena con DPO y mide la recompensa verdadera. ¿Qué ocurre con «por favor»?
3. **REINFORCE en la política diminuta.** Optimiza `pol` directamente con REINFORCE + línea base + penalización KL, usando `recompensa` como modelo de recompensa. Compara con DPO.

## Soluciones de referencia

In [ ]:
# Ejercicio 1
# def grad_log_pi(theta, a):
#     g = -softmax(theta)
#     g[a] += 1
#     return g

# Ejercicio 2 (dentro del bucle)
#     ventaja = r - b if linea_base else r
#     theta = theta + lr * ventaja * grad_log_pi(theta, a)

# Ejercicio 3 (dentro del bucle)
#     G[i] = (r - b) * grad_log_pi(theta, a)

# Ejercicio 4
# def bt_perdida_grad(w, Xw, Xl):
#     D = Xw - Xl
#     m = D @ w
#     perdida = np.mean(np.logaddexp(0, -m))
#     grad = -((1 - sigmoide(m))[:, None] * D).mean(axis=0)
#     return perdida, grad

# Ejercicio 5
# def perdida_dpo(pol, ref, x, yw, yl, beta):
#     hw = pol.logp(x, yw) - ref.logp(x, yw)
#     hl = pol.logp(x, yl) - ref.logp(x, yl)
#     margen = beta * (hw - hl)
#     return -F.logsigmoid(margen).mean(), (margen > 0).float().mean()

# Ejercicio 6 (dentro del bucle por prompt)
#         pr = ref.logp(X, TODAS).exp()
#         pp = pol.logp(X, TODAS).exp()
#         po = pr * torch.exp(R_TODAS / BETA); po = po / po.sum()
#         Er_ref.append((pr * R_TODAS).sum().item())
#         Er_pol.append((pp * R_TODAS).sum().item())
#         Er_opt.append((po * R_TODAS).sum().item())